# Ham ya da Spam?

🎯 Bu görevin amacı, e-postaları **spam (1)** veya **normal e-posta (0)** olarak sınıflandırmaktır.

🧹 İlk olarak, bu metin verilerine **temizleme (cleaning)** teknikleri uygulanacaktır.

👩🏻‍🔬 Ardından, temizlenmiş metinler **sayısal bir gösterime** dönüştürülecektir.

✉️ Son olarak, her bir e-postayı spam mı yoksa normal mi olduğunu sınıflandırmak için  
***Multinomial Naive Bayes*** modeli uygulanacaktır.


## (0) NTLK kütüphanesi (Doğal Dil Araç Seti)

In [2]:
#!pip install nltk

In [3]:
# nltk'yi ilk kez içe aktarırken, birkaç yerleşik kütüphaneyi de indirmemiz gerekir.

import nltk

nltk.download('stopwords')
nltk.download('punkt')      # nltk<3.9.0 için
nltk.download('punkt_tab')  # nltk>=3.9.0 için
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /Users/gonul/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /Users/gonul/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /Users/gonul/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /Users/gonul/nltk_data...
[nltk_data] Downloading package omw-1.4 to /Users/gonul/nltk_data...


True

In [4]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/ham_spam_emails.csv")
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


## (1) (Metin) veri setinin temizlenmesi

Veri kümesi, ham [0] veya spam [1] olarak sınıflandırılan e-postalardan oluşur. Tahmin modelini eğitmeden önce veri kümesini temizlemeniz gerekir.

### (1.1) Noktalama İşaretlerini Kaldır

❓ Noktalama işaretlerini kaldırmak için bir işlev oluşturun. Bunu `text` sütununa uygulayın ve çıktıyı `clean_text` adlı veri çerçevesinin yeni bir sütununa ekleyin. ❓

In [20]:
# SENİN KODUN BURAYA
import string

def remove_punctuation(text):
    return text.translate(
        str.maketrans('', '', string.punctuation)
    )
    
df["clean_text"] = df["text"].apply(remove_punctuation)

### (1.2) Küçük Harf

❓ Metni küçük harfe çeviren bir işlev oluşturun. Bunu `clean_text`'e uygulayın ❓

In [21]:
# SENİN KODUN BURAYA
df["clean_text"] = df["text"].str.replace(r'[^\w\s]', '', regex=True).str.lower()
df.head()


,text,spam,clean_text
0,Subject: naturally irresistible your corporate...,1,subject naturally irresistible your corporate ...
1,Subject: the stock trading gunslinger fanny i...,1,subject the stock trading gunslinger fanny is...
2,Subject: unbelievable new homes made easy im ...,1,subject unbelievable new homes made easy im w...
3,Subject: 4 color printing special request add...,1,subject 4 color printing special request addi...
4,"Subject: do not have money , get software cds ...",1,subject do not have money get software cds fr...


In [ ]:
# df.loc[0, "clean_text"]

### (1.3) Sayıları Kaldır

❓ Metinden sayıları kaldırmak için bir işlev oluşturun. Bunu `clean_text`'e uygulayın ❓

In [22]:
# SENİN KODUN BURAYA
def remove_numbers(text):
    return text.translate(
        str.maketrans('', '', string.digits)
    )
df["clean_text"] = df["clean_text"].apply(remove_numbers)


### (1.4) StopWords'ü kaldırın

❓ Metinden durdurma kelimelerini kaldırmak için bir işlev oluşturun. Bunu `clean_text`'e uygulayın. ❓

In [23]:
# SENİN KODUN BURAYA
stop_words = set(nltk.corpus.stopwords.words('english'))

def remove_stopwords(text, stop_words=stop_words):
    tokens = nltk.word_tokenize(text)
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(filtered_tokens)

df["clean_text"] = df["clean_text"].apply(remove_stopwords)

### (1.5) Lemmatize

❓ Metni lemmatize etmek için bir fonksiyon oluşturun. Çıktının bir kelime listesi değil, tek bir dize olduğundan emin olun. Bunu `clean_text`'e uygulayın. ❓

In [24]:
# SENİN KODUN BURAYA
lemmatizer = nltk.stem.WordNetLemmatizer()
def lemmatize_text(text, lemmatizer=lemmatizer):
    tokens = nltk.word_tokenize(text)
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(lemmatized_tokens)

df["clean_text"] = df["clean_text"].apply(lemmatize_text)

## (2) Bag-of-Words Modellemesi

### (2.1) Metin verilerini sayılara dönüştürme

❓ `clean_text`'i varsayılan CountVectorizer ile Bag-of-Words temsiline vektörleştirin. `X_bow` olarak kaydedin. ❓

In [ ]:
# SENİN KODUN BURAYA
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(df["clean_text"])


In [32]:
X_bow[0].toarray()

array([[0, 0, 0, ..., 0, 0, 0]])

In [33]:
vectorizer.get_feature_names_out()

array(['aa', 'aaa', 'aaaenerfax', ..., 'zzn', 'zzncacst', 'zzzz'],
      dtype=object)

In [34]:
vectorizer.get_feature_names_out()[:20]

array(['aa', 'aaa', 'aaaenerfax', 'aadedeji', 'aagrawal', 'aal',
       'aaldous', 'aaliyah', 'aall', 'aanalysis', 'aaron', 'aawesome',
       'ab', 'aba', 'abacha', 'abacus', 'abahy', 'abaixo', 'abandon',
       'abandoned'], dtype=object)

### (2.2) Çok terimli Naive Bayes Modellemesi

❓ MultinomialNB modelini bag-of-words verileriyle çapraz doğrulayın. Modelin doğruluğunu puanlayın. ❓

In [36]:
# SENİN KODUN BURAYA
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_validate

model = MultinomialNB()

cv_results = cross_validate(model, X_bow, df["spam"], cv=5, scoring=["accuracy", "precision", "recall", "f1"])
cv_results


{'fit_time': array([0.00700068, 0.00396705, 0.00258207, 0.00217128, 0.00227618]),
 'score_time': array([0.00495005, 0.00332189, 0.00285578, 0.00376296, 0.00289512]),
 'test_accuracy': array([0.98691099, 0.9895288 , 0.991274  , 0.98777293, 0.99213974]),
 'test_precision': array([0.95438596, 0.96785714, 0.97826087, 0.95438596, 0.97482014]),
 'test_recall': array([0.99270073, 0.98905109, 0.98540146, 0.996337  , 0.99267399]),
 'test_f1': array([0.97316637, 0.97833935, 0.98181818, 0.97491039, 0.98366606])}

🏁 Tebrikler!

💾 Not defterinizi git add/commit/push yapmayı unutmayın...

🚀 ... ve bir sonraki challenge'a geçin!